# DATA READING

In [ ]:
df = (spark.read.format("parquet")
      .option("inferSchema", True)
      .load("abfss://bronze@emilyscardatalake.dfs.core.windows.net/rawdata/"))

df.printSchema()
df.display()

# DATA TRANSFORMATION

In [ ]:
from pyspark.sql.functions import *

In [ ]:
df = df.withColumn('model_category', split(col("Model_ID"),'-')[0])
display(df)

# AD-HOC

In [ ]:
df.groupBy('Year','BranchName').agg(sum('Units_Sold').alias('Total_Units')).sort('Year', 'Total_Units', ascending=[1,0]).display()

In [ ]:
%sql
GRANT READ FILES ON EXTERNAL LOCATION bronzeext TO `t***g98@gmail.com`;

In [ ]:
df_files = (spark.read.format("parquet")
            .option("inferSchema", True)
            .load("abfss://bronze@emilyscardatalake.dfs.core.windows.net/rawdata/"))

df_files.select("_metadata.file_path").distinct().show(truncate=False)

# DATA WRITING

In [ ]:
df.write.format('parquet')\
        .mode('overwrite')\
        .option('path','abfss://silver@emilyscardatalake.dfs.core.windows.net/carsales')\
        .save()

# QUERYING SILVER DATA

%sql
SELECT * FROM parquet.`abfss://silver@emilyscardatalake.dfs.core.windows.net/carsales`